# AGU RAG — Colab GPU Embedding

Bu notebook `data/processed/*.jsonl` chunk dosyalarini **T4 GPU** ile embed edip `data/chroma/` indeksini uretir; en sonda `chroma.zip` olarak indirir.

## Calistirma sirasi
1. **Runtime -> Change runtime type -> T4 GPU** sectiginden emin ol.
2. Hucreleri tek tek Shift+Enter ile calistir.
3. Son hucrede `chroma.zip` otomatik inecek.
4. Yerelde mevcut `data/chroma/` klasorunu silip zip'i `data/` icine ac.

In [ ]:
# 1) GPU kontrolu — T4 (veya benzeri) gorunmeli
!nvidia-smi

In [ ]:
# 2) Repo'yu klonla
# NOT: Repo PRIVATE ise REPO_URL'i 'https://<TOKEN>@github.com/alibakir01/RAG-asistan.git' yap.
import os
REPO_URL = 'https://github.com/alibakir01/RAG-asistan.git'
REPO_DIR = '/content/RAG-asistan'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
%cd {REPO_DIR}
!ls -la data/processed/

In [ ]:
# 3) Bagimliliklari kur (streamlit/groq gerekmez — sadece embed icin)
# torch+CUDA Colab'da hazir geldigi icin yeniden kurulmuyor.
!pip install -q sentence-transformers chromadb

In [ ]:
# 4) Chunk dosyalarini dogrula
from pathlib import Path
p = Path('data/processed')
files = sorted(p.glob('*.jsonl'))
total = 0
for f in files:
    n = sum(1 for _ in f.open(encoding='utf-8'))
    total += n
    print(f'  {f.name:40s} {n:>5d} chunk')
print(f'\nTOPLAM: {total} chunk')

In [ ]:
# 5) CUDA dogrulama
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
print(f'CUDA device    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'PyTorch        : {torch.__version__}')
assert torch.cuda.is_available(), 'GPU bulunamadi! Runtime > Change runtime type > T4 GPU sec.'

In [ ]:
# 6) Embedding — embed.py SentenceTransformer otomatik CUDA secer
# T4'te ~500-1000 chunk icin tahmini sure: 30-90 sn
import time
t0 = time.time()
!python src/embed.py
print(f'\nToplam sure: {time.time()-t0:.1f} sn')

In [ ]:
# 7) Chroma indeksini hizlica test et
import chromadb
from sentence_transformers import SentenceTransformer
client = chromadb.PersistentClient(path='data/chroma')
col = client.get_collection('agu_comp')
print(f'Indekslenen chunk sayisi: {col.count()}')
model = SentenceTransformer('intfloat/multilingual-e5-large')
q = model.encode(['query: 2023 girisliyim 3. donem hangi dersler var'], normalize_embeddings=True).tolist()
r = col.query(query_embeddings=q, n_results=3)
for i, doc in enumerate(r['documents'][0], 1):
    print(f'\n[{i}] {doc[:200]}...')

In [ ]:
# 8) Chroma klasorunu ZIP'le
import shutil, os
out = shutil.make_archive('/content/chroma', 'zip', 'data', 'chroma')
size_mb = os.path.getsize(out) / 1024 / 1024
print(f'{out} — {size_mb:.1f} MB hazir')

In [ ]:
# 9) Tarayicidan indir
from google.colab import files
files.download('/content/chroma.zip')

## Yerelde kurulum (PowerShell)

```powershell
# 1) Mevcut chroma'yi sil
Remove-Item -Recurse -Force data\chroma

# 2) Indirdigin chroma.zip'i proje koklerine koy, sonra ac:
Expand-Archive -Path chroma.zip -DestinationPath data\

# 3) Streamlit'i baslat
streamlit run app.py
```

## Notlar
- `data/processed/` icindeki chunk dosyalari GitHub repo'da olmali. Untracked dosyalar (orn. `chunks_siyaset.jsonl`) once commit/push edilmeli, yoksa Colab onlari gormez.
- Repo private ise klonlama hucresinde token kullan: `https://ghp_xxx@github.com/...`
- Embedding'i tekrar calistirmadan once eski `data/chroma/`'yi silmeyi unutma — `embed.py` zaten siliyor ama yerel ortamla karistirma.